SymPy (https://docs.sympy.org/latest/guides/solving/index.html) can symbolically solve equations, differential equations, linear equations, nonlinear equations, matrix problems, inequalities, Diophantine equations, and evaluate integrals. SymPy can also solve numerically.
https://www.sympy.org/scipy-2017-codegen-tutorial/notebooks/01-intro-sympy.html
These examples show how SymPy:<br>
1. Solve an equation algebraically
2. Solve a system of equations algebraically
3. Solve one or a system of equations numerically
4. Solve an ordinary differential equation algebraically
5. Find the roots of a polynomial algebraically or numerically
6. Solve a matrix equation algebraically
7. Reduce one or a system of inequalities for a single variable algebraically
8. Solve a Diophantine equation algebraically

Notes:
SymPy has a function called solve() which is designed to find the solutions of an equation or system of equations, or the roots of a function. While a common, colloquial expression is, for example, “solve an integral,” in SymPy’s terminology it would be “evaluate an integral.” 

Solving Guidance<br>
These guidelines apply to many types of solving.

Numeric Solutions - The vast majority of arbitrary nonlinear equations have no closed-form solution. The classes of equations that are solvable are basically:

1. Linear equations
2. Polynomials, except where limited by the Abel-Ruffini theorem (learn more about solving polynomials using a GroebnerBasis)
3. Equations that can be solved by inverting some transcendental functions
4. Problems that can be transformed into the cases above (e.g., by turning trigonometric functions into polynomials)
5. A few other special cases that can be solved with something like the Lambert W function
6. Equations that you can decompose() via any of the above

SymPy may reflect that your equation has no solutions that can be expressed algebraically (symbolically), or that SymPy lacks an algorithm to find a closed-form solution that does exist, by returning an error such as NotImplementedError:

In [ ]:
from sympy import solve, cos
from sympy.abc import x
solve(cos(x) - x, x, dict=True)

# I) Solve an Equation Algebraically  (symbolically) 
There are two high-level functions to solve equations, solve() and solveset(). 

In [2]:
from sympy.abc import x, y
from sympy import solve
solve(x**2 - y, x, dict=True)

[{x: -sqrt(y)}, {x: sqrt(y)}]

In [4]:
from sympy import solveset
from sympy.abc import x, y
solveset(x**2 - y, x)

{-sqrt(y), sqrt(y)}

In [6]:
from sympy import Eq, solve, solveset
from sympy.abc import x, y
eqn = Eq(x**2, y)
eqn
solutions = solve(eqn, x, dict=True)
print(solutions)
solutions_set = solveset(eqn, x)
print(solutions_set)
for solution_set in solutions_set:
    print(solution_set)

[{x: -sqrt(y)}, {x: sqrt(y)}]
{-sqrt(y), sqrt(y)}
sqrt(y)
-sqrt(y)


Restrict the Domain of Solutions
By default, SymPy will return solutions in the complex domain, which also includes purely real and imaginary values. Here, the first two solutions are real, and the last two are imaginary:

In [ ]:
import sympy
from sympy import Symbol, solve, solveset
x = Symbol('x')
solve(x**4 - 256, x, dict=True)
solveset(x**4 - 256, x)

Restrict returned solutions to real numbers, or another domain or range, the different solving functions use different methods.

For solve(), place an assumption on the symbol to be solved for

In [1]:
from sympy import Symbol, solve
x = Symbol('x', real=True)
solve(x**4 - 256, x, dict=True)

[{x: -4}, {x: 4}]

or restrict the solutions with standard Python techniques for filtering a list such as a list comprehension:

In [7]:
from sympy import Or, Symbol, solve
x = Symbol('x', real=True)
expr = (x-4)*(x-3)*(x-2)*(x-1)
solution = solve(expr, x)
print(solution)
solution_outside_2_3 = [v for v in solution if (v.is_real and Or(v<2,v>3))]
print(solution_outside_2_3)

[1, 2, 3, 4]
[1, 4]


In [8]:
# For solveset(), limit the output domain in the function call by setting a domain

from sympy import S, solveset
from sympy.abc import x
solveset(x**4 - 256, x, domain=S.Reals)

# or by restricting returned solutions to any arbitrary set, including an interval:

from sympy import Interval, pi, sin, solveset
from sympy.abc import x
solveset(sin(x), x, Interval(-pi, pi))

{0, -pi, pi}

In [9]:
# if you restrict the solutions to a domain in which there are no solutions, solveset() will return the empty set, EmptySet:

from sympy import solveset, S
from sympy.abc import x
solveset(x**2 + 1, x, domain=S.Reals)

EmptySet


Explicitly Represent Infinite Sets of Possible Solutions
solveset() can represent infinite sets of possible solutions and express them in standard mathematical notation, for example  for every integer value of :

In [10]:

from sympy import pprint, sin, solveset
from sympy.abc import x
solution = solveset(sin(x), x)
pprint(solution)

{2⋅n⋅π │ n ∊ ℤ} ∪ {2⋅n⋅π + π │ n ∊ ℤ}


In [11]:

# However, solve() will return only a finite number of solutions: solve() tries to return just enough solutions so that all (infinitely many) solutions can generated from the returned solutions by adding integer multiples of the periodicity() of the equation

from sympy import sin, solve
from sympy.calculus.util import periodicity
from sympy.abc import x
f = sin(x)
solve(f, x)
periodicity(f, x)

2*pi

Use the Solution Result. Substitute Solutions From solve() Into an Expression
You can substitute solutions from solve() into an expression.

A common use case is finding the critical points and values for a function . At the critical points, the Derivative equals zero (or is undefined). You can then obtain the function values at those critical points by substituting the critical points back into the function using subs(). You can also tell if the critical point is a maxima or minima by substituting the values into the expression for the second derivative: a negative value indicates a maximum, and a positive value indicates a minimum.

In [12]:
from sympy.abc import x
from sympy import solve, diff
f = x**3 + x**2 - x
derivative = diff(f, x)
critical_points = solve(derivative, x, dict=True)
print(critical_points)
point1, point2 = critical_points
print(f.subs(point1))
print(f.subs(point2))
curvature = diff(f, x, 2)
print(curvature.subs(point1))
print(curvature.subs(point2))

[{x: -1}, {x: 1/3}]
1
-5/27
-4
4


solveset() Solution Sets Cannot Necessarily Be Interrogated Programmatically
If solveset() returns a finite set (class FiniteSet), you can iterate through the solutions:

In [13]:
from sympy import solveset
from sympy.abc import x, y
solution_set = solveset(x**2 - y, x)
print(solution_set)
solution_list = list(solution_set)
print(solution_list)

{-sqrt(y), sqrt(y)}
[sqrt(y), -sqrt(y)]


In [14]:
# However, for more complex results, it may not be possible to list the solutions:

from sympy import S, solveset, symbols, intersection, sqrt
x, y = symbols('x, y')
solution_set = solveset(x**2 - y, x, domain=S.Reals)
print(solution_set)
intersection({-sqrt(y), sqrt(y)}, domain=S.Reals)
list(solution_set)

Intersection({-sqrt(y), sqrt(y)}, Reals)


TypeError: The computation had not completed because of the undecidable set membership is found in every candidates.

TypeError: The computation had not completed because of the undecidable set
membership is found in every candidates.

In this case, it is because, if  is negative, its square root would be imaginary rather than real and therefore outside the declared domain of the solution set. By declaring  to be real and positive, SymPy can determine that its square root is real, and thus resolve the intersection between the solutions and the set of real numbers:

In [15]:
from sympy import S, Symbol, solveset
x = Symbol('x')
y = Symbol('y', real=True, positive=True)
solution_set = solveset(x**2 - y, x, domain=S.Reals)
print(solution_set)

list(solution_set)

{-sqrt(y), sqrt(y)}


[sqrt(y), -sqrt(y)]

In [ ]:
# Alternatively, you can extract the sets from the solution set using args, then create a list from the set containing the symbolic solutions:

from sympy import S, solveset, symbols, intersection
x, y = symbols('x, y')
solution_set = solveset(x**2 - y, x, domain=S.Reals)
print(solution_set)
intersection({-sqrt(y), sqrt(y)}, domain=S.Reals)
solution_set_args = solution_set.args
print(solution_set.args)

list(solution_set_args[1])

Not All Equations Can Be Solved. Equations With No Closed-Form Solution
Some equations have no closed-form solution, in which case SymPy may return an empty set or give an error. For example, the following transcendental equation has no closed-form solution:

Equations Which Have a Closed-Form Solution, and SymPy Cannot Solve. 
It is also possible that there is an algebraic solution to your equation, and SymPy has not implemented an appropriate algorithm. 

In [ ]:
from sympy import cos, solve
from sympy.abc import x
solve(cos(x) - x, x, dict=True)

# II) Solve a System of Equations Algebraically

linear or nonlinear. For example, solving $x^2 + y = 2z, y = -4z$ for x and y (assuming z
is a constant or parameter) yields $\{(x = -\sqrt{6z}, y = -4z),$ ${(x =
\sqrt{6z}, y = -4z)\}}$.

- Some systems of equations cannot be solved algebraically (either at all or by SymPy), so you may have to [solve your system of equations numerically](solve-numerically.md) using {func}`~.nsolve` instead.

Whether your equations are linear or nonlinear, you can use {func}`~.solve`:

### Solve a System of Linear Equations Algebraically

In [7]:
from sympy import solve
from sympy.abc import x, y, z
solve([x + y - 2*z, y + 4*z], [x, y], dict=True)
# solve([x + y - 2*z, y + 4*z], [x, y])

[{x: 6*z, y: -4*z}]

### Solve a System of Nonlinear Equations Algebraically

In [5]:
from sympy import solve
from sympy.abc import x, y, z
solve([x**2 + y - 2*z, y + 4*z], x, y, dict=True)

[{x: -sqrt(6)*sqrt(z), y: -4*z}, {x: sqrt(6)*sqrt(z), y: -4*z}]

Guidance : Refer to
[](solving-guidance.md#include-the-variable-to-be-solved-for-in-the-function-call)
and [](ensure-consistent-formatting-from-solve).

There are two methods below for containing solution results:
[dictionary](#solve-and-use-results-in-a-dictionary) or
[set](#solve-results-in-a-set). A dictionary is easier to interrogate
programmatically, so if you need to extract solutions using code, we recommend
the dictionary approach.

### Solve and Use Results in a Dictionary

Solve Into a Solution Given as a Dictionary

You can solve a system of equations for some variables (for example, $x$ and
$y$) leaving another symbol as a constant or parameter (for example, $z$). You
can specify the variables to solve for as multiple separate arguments, or as a
list (or tuple):

In [16]:
from sympy import solve
from sympy.abc import x, y, z
equations = [x**2 + y - 2*z, y + 4*z]
solutions = solve(equations, x, y, dict=True)
solutions

[{x: -sqrt(6)*sqrt(z), y: -4*z}, {x: sqrt(6)*sqrt(z), y: -4*z}]

Use a Solution Given as a Dictionary

You can then extract solutions by indexing (specifying in brackets) the solution
number, and then the symbol. For example `solutions[0][x]` gives the result for
`x` in the first solution:

In [ ]:
solutions[0][x]
solutions[0][y]

-4*z

Solve Results in a Set

To get a list of symbols and set of solutions, use `set=True` instead of
`dict=True`:

In [2]:
from sympy import solve
from sympy.abc import x, y, z
solve([x**2 + y - 2*z, y + 4*z], [x, y], set=True)
# ([x, y], {(-sqrt(6)*sqrt(z), -4*z), (sqrt(6)*sqrt(z), -4*z)})

([x, y], {(-sqrt(6)*sqrt(z), -4*z), (sqrt(6)*sqrt(z), -4*z)})


Options That Can Speed up {func}`~.solve`. Refer to [](options-that-can-speed-up-solve).

Not All Systems of Equations Can be Solved. Systems of Equations With no Solution
e.g., the following two systems have no solution because they reduce to `1 == 0`, so SymPy returns an empty list:

In [1]:
from sympy import solve
from sympy.abc import x, y
solve([x + y - 1, x + y], [x, y], dict=True)

[]

In [3]:

from sympy import solve
from sympy.abc import x, y, z
solve([x + y - (z + 1), x + y - z], [x, y], dict=True)

[]


The following system reduces to $z = 2z$, so it has no general solution, but it
could be satisfied if $z=0$. Note that {func}`~.solve` will not assume that
$z=0$, even though that is the only value of $z$ that makes the system of
equations consistent, because $z$ is a parameter rather than an unknown. That
is, {func}`~.solve` does not treat $z$ as an unknown because it is not in the
list of symbols specified as unknowns (`[x, y]`) and all such symbols are
treated like parameters with arbitrary value. Whether a symbol is treated as a
variable or a parameter is determined only by whether it is specified as a
symbol to solve for in {func}`~.solve`. There is no such distinction made when
creating the symbol using {func}`~.symbols` (or importing from {mod}`~.abc`).

In [4]:
from sympy import solve
from sympy.abc import x, y, z
solve([x + y - z, x + y - 2*z], [x, y], dict=True)

[]

The following system is [overconstrained](https://en.wikipedia.org/wiki/Overdetermined_system), meaning there are more equations (three) than unknowns to be solved for (two, namely $x$ and $y$). It has no solution:

In [16]:
from sympy import solve
from sympy.abc import x, y, z
solve([x + y - z, x - (z + 1), 2*x - y], [x, y], dict=True)

[]

Note that some overconstrained systems do have solutions (for example, if an
equation is a linear combination of the others), in which case SymPy can solve
the overconstrained system.

Systems of Equations With no Closed-Form Solution

Some systems of equations cannot be solved algebraically, for example those
containing [transcendental equations](https://en.wikipedia.org/wiki/Transcendental_equation):


In [ ]:
from sympy import cos, solve
from sympy.abc import x, y, z
solve([x - y, cos(x) - y], [x, y], dict=True)

So you can use {func}`~.nsolve` to [find a numerical
solution](solve-numerically.md):

In [ ]:
from sympy import cos, nsolve
from sympy.abc import x, y, z
nsolve([x - y, cos(x) - y], [x, y], [1,1])

Equations Which Have a Closed-Form Solution, and SymPy Cannot Solve

It is also possible that there is an algebraic solution to your equation, and
SymPy has not implemented an appropriate algorithm. If SymPy returns an empty
set or list when you know there is a closed-form solution (indicating a bug in
SymPy), please post it on the [mailing list](https://groups.google.com/g/sympy),
or open an issue on [SymPy's GitHub page](https://github.com/sympy/sympy/issues). Until the issue is resolved, you can use a different method listed in [](#alternatives-to-consider).

# III) Solve One or a System of Equations Numerically
Solving numerically is useful if:

You only need a numeric solution, not a symbolic one

A closed-form solution is not available or is overly complicated; refer to When You Might Prefer a Numeric Solution

solve() and solveset() will not try to find a numeric solution, only a mathematically-exact symbolic solution. So if you want a numeric solution, use nsolve().

SymPy is designed for symbolic mathematics. If you do not need to do symbolic operations, then for numerical operations you can use another free and open-source package such as NumPy or SciPy which will be faster, work with arrays, and have more algorithms implemented. The main reasons to use SymPy (or its dependency mpmath) for numerical calculations are:

to do a simple numerical calculation within the context of a symbolic calculation using SymPy

if you need the arbitrary precision capabilities to get more digits of precision than you would get from float64.

Alternatives to Consider
SciPy’s scipy.optimize.fsolve() can solve a system of (non-linear) equations

NumPy’s numpy.linalg.solve() can solve a system of linear scalar equations

mpmath’s findroot(), which nsolve() calls and can pass parameters to
 


Example of Numerically Solving an Equation

In [5]:
from sympy import cos, nsolve, Symbol
x = Symbol('x')
nsolve(cos(x) - x, x, 1)

0.739085133215161

Overdetermined systems of equations are supported.

Find Complex Roots of a Real Function
To solve for complex roots of real functions, specify a nonreal (either purely imaginary, or complex) initial point:

In [ ]:
from sympy import nsolve
from sympy.abc import x
nsolve(x**2 + 2, 1) # Real initial point returns no root

In [4]:
# Try another starting point or tweak arguments.
from sympy import I
nsolve(x**2 + 2, I) # Imaginary initial point returns a complex root

1.4142135623731*I

In [6]:
nsolve(x**2 + 2, 1 + I) # Complex initial point returns a complex root

1.4142135623731*I

Ensure the Root Found is in a Given Interval
It is not guaranteed that nsolve() will find the root closest to the initial point. Here, even though the root -1 is closer to the initial point of -0.1, nsolve() finds the root 1:

In [7]:
from sympy import nsolve
from sympy.abc import x
nsolve(x**2 - 1, -0.1)

1.00000000000000

You can ensure the root found is in a given interval, if such a root exists, using solver='bisect' by specifying the interval in a tuple. Here, specifying the interval (-10, 0) ensures that the root -1 is found:

In [8]:
from  sympy import nsolve
from sympy.abc import x
nsolve(x**2 - 1, (-10, 0), solver='bisect')

-1.00000000000000

Solve a System of Equations Numerically
To solve a system of multidimensional functions, supply a tuple of

functions (f1, f2)

variables to solve for (x1, x2)

starting values (-1, 1)

In [ ]:
from sympy import Symbol, nsolve
x1 = Symbol('x1')
x2 = Symbol('x2')
f1 = 3 * x1**2 - 2 * x2**2 - 1
f2 = x1**2 - 2 * x1 + x2**2 + 2 * x2 - 8
print(nsolve((f1, f2), (x1, x2), (-1, 1)))

#Increase Precision of the Solution
# You can increase the precision of the solution using prec:

from sympy import Symbol, nsolve
x1 = Symbol('x1')
x2 = Symbol('x2')
f1 = 3 * x1**2 - 2 * x2**2 - 1
f2 = x1**2 - 2 * x1 + x2**2 + 2 * x2 - 8
print(nsolve((f1, f2), (x1, x2), (-1, 1), prec=25))

Matrix([[-1.19287309935246], [1.27844411169911]])
Matrix([[-1.192873099352460791205211], [1.278444111699106966687122]])


Create a Function That Can Be Solved With SciPy
As noted above, SymPy focuses on symbolic computation and is not optimized for numerical calculations. If you need to make many calls to a numerical solver, it can be much faster to use a solver optimized for numerical calculations such as SciPy’s root_scalar(). A recommended workflow is:

use SymPy to generate (by symbolically simplifying or solving an equation) the mathematical expression

convert it to a lambda function using lambdify()

use a numerical library such as SciPy to generate numerical solutions

In [7]:
from sympy import simplify, cos, sin, lambdify
from sympy.abc import x, y
from scipy.optimize import root_scalar
expr = cos(x * (x + x**2)/(x*sin(y)**2 + x*cos(y)**2 + x))
simplify(expr) # 1. symbolically simplify expression
cos(x*(x + 1)/2)
lam_f = lambdify(x, cos(x*(x + 1)/2)) # 2. lambdify
sol = root_scalar(lam_f, bracket=[0, 2]) # 3. numerically solve using SciPy
sol.root

1.3416277185114782

Use the Solution Result - Substitute the Result Into an Expression
The best practice is to use evalf() to substitute numerical values into expressions. The following code demonstrates that the numerical value is not an exact root because substituting it back into the expression produces a result slightly different from zero:

In [ ]:

from sympy import cos, nsolve, Symbol
x = Symbol('x')
f = cos(x) - x
x_value = nsolve(f, x, 1); x_value


-5.12757857962640e-17

In [10]:
f.evalf(subs={x: x_value})

-5.12757857962640e-17

Using subs can give an incorrect result due to precision errors, here effectively rounding -5.12757857962640e-17 to zero:

In [ ]:
f.subs(x, x_value)

In [11]:
# When substituting in values, you can also leave some symbols as variables:

from sympy import cos, nsolve, Symbol
x = Symbol('x')
f = cos(x) - x
x_value = nsolve(f, x, 1); x_value
0.739085133215161
y = Symbol('y')
z = Symbol('z')
g = x * y**2
values = {x: x_value, y: 1}
(x + y - z).evalf(subs=values)

1.73908513321516 - z

Not all Equations Can be Solved
nsolve() is a numerical solving function, so it can often provide a solution for equations which cannot be solved algebraically.

Equations With no Solution
Some equations have no solution, in which case SymPy may return an error. For example, the equation (exp(x) in SymPy) has no solution:

In [ ]:
from sympy import nsolve, exp
from sympy.abc import x
nsolve(exp(x), x, 1, prec=20)

## Solve an Ordinary Differential Equation (ODE) Algebraically
Use SymPy to solve an ordinary differential equation (ODE) algebraically.
Alternatives to Consider - To numerically solve a system of ODEs, use a SciPy ODE solver such as solve_ivp. You can also use SymPy to create and then lambdify() an ODE to be solved numerically using SciPy’s as solve_ivp as described below in Numerically Solve an ODE in SciPy.

Solve an Ordinary Differential Equation (ODE)
Here is an example of solving the above ordinary differential equation algebraically using dsolve(). You can then use checkodesol() to verify that the solution is correct.

In [ ]:
from sympy import Function, dsolve, Derivative, checkodesol
from sympy.abc import x
y = Function('y')
# Solve the ODE
result = dsolve(Derivative(y(x), x, x) + 9*y(x), y(x))
result

Eq(y(x), C1*sin(3*x) + C2*cos(3*x))

In [4]:
# Check that the solution is correct
checkodesol(Derivative(y(x), x, x) + 9*y(x), result)

(True, 0)

The output of checkodesol() is a tuple where the first item, a boolean, tells whether substituting the solution into the ODE results in 0, indicating the solution is correct.

Defining Derivatives - There are many ways to express derivatives of functions. For an undefined function, both Derivative and diff() represent the undefined derivative. Thus, all of the following ypp (“y prime prime”) represent 
, the second derivative with respect to of a function 

In [ ]:
ypp = y(x).diff(x, x)
ypp = y(x).diff(x, 2)
ypp = y(x).diff((x, 2))
ypp = diff(y(x), x, x)
ypp = diff(y(x), x, 2)
ypp = Derivative(y(x), x, x)
ypp = Derivative(y(x), x, 2)
ypp = Derivative(Derivative(y(x), x), x)
ypp = diff(diff(y(x), x), x)
yp = y(x).diff(x)
ypp = yp.diff(x)

We recommend specifying the function to be solved for, as the second argument to dsolve(). Note that it must be a function rather than a variable (symbol). SymPy will give an error if you specify a variable () rather than a function ():

In [ ]:
dsolve(Derivative(y(x), x, x) + 9*y(x), x)

Similarly, you must specify the argument of the function: 
, not just .

Options to Define an ODE - You can define the function to be solved for in two ways. The subsequent syntax for specifying initial conditions depends on your choice.

Option 1: Define a Function Without Including Its Independent Variable
You can define a function without including its independent variable:

In [7]:
from sympy import symbols, Eq, Function, dsolve
f, g = symbols("f g", cls=Function)
x = symbols("x")
eqs = [Eq(f(x).diff(x), g(x)), Eq(g(x).diff(x), f(x))]
dsolve(eqs, [f(x), g(x)])

[Eq(f(x), -C1*exp(-x) + C2*exp(x)), Eq(g(x), C1*exp(-x) + C2*exp(x))]

*
Note that you supply the functions to be solved for as a list as the second argument of dsolve(), here [f(x), g(x)].

Specify Initial Conditions or Boundary Conditions
If your differential equation(s) have initial or boundary conditions, specify them with the dsolve() optional argument ics. Initial and boundary conditions are treated the same way (even though the argument is called ics). It should be given in the form of {f(x0): y0, f(x).diff(x).subs(x, x1): y1} and so on where, for example, the value of 
 at 
 is 
. For power series solutions, if no initial conditions are specified 
 is assumed to be 
 and the power series solution is calculated about 
.

Here is an example of setting the initial values for functions, namely namely 
 and 
:

from sympy import symbols, Eq, Function, dsolve
f, g = symbols("f g", cls=Function)
x = symbols("x")
eqs = [Eq(f(x).diff(x), g(x)), Eq(g(x).diff(x), f(x))]
dsolve(eqs, [f(x), g(x)])
[Eq(f(x), -C1*exp(-x) + C2*exp(x)), Eq(g(x), C1*exp(-x) + C2*exp(x))]
dsolve(eqs, [f(x), g(x)], ics={f(0): 1, g(2): 3})
[Eq(f(x), (1 + 3*exp(2))*exp(x)/(1 + exp(4)) - (-exp(4) + 3*exp(2))*exp(-x)/(1 + exp(4))), Eq(g(x), (1 + 3*exp(2))*exp(x)/(1 + exp(4)) + (-exp(4) + 3*exp(2))*exp(-x)/(1 + exp(4)))]
Here is an example of setting the initial value for the derivative of a function, namely 
:

eqn = Eq(f(x).diff(x), f(x))
dsolve(eqn, f(x), ics={f(x).diff(x).subs(x, 1): 2})
Eq(f(x), 2*exp(-1)*exp(x))
Option 2: Define a Function of an Independent Variable
You may prefer to specify a function (for example 
) of its independent variable (for example 
), so that y represents y(t):

from sympy import symbols, Function, dsolve
t = symbols('t')
y = Function('y')(t)
y
y(t)
yp = y.diff(t)
ypp = yp.diff(t)
eq = ypp + 2*yp + y
eq
y(t) + 2*Derivative(y(t), t) + Derivative(y(t), (t, 2))
dsolve(eq, y)
Eq(y(t), (C1 + C2*t)*exp(-t))
Using this convention, the second argument of dsolve(), y, represents y(t), so SymPy recognizes it as a valid function to solve for.

Specify Initial Conditions or Boundary Conditions
Using that syntax, you specify initialor boundary conditions by substituting in values of the independent variable using subs() because the function 
 already has its independent variable as an argument 
:

dsolve(eq, y, ics={y.subs(t, 0): 0})
Eq(y(t), C2*t*exp(-t))
Beware Copying and Pasting Results
If you choose to define a function of an independent variable, note that copying a result and pasting it into subsequent code may cause an error because x is already defined as y(t), so if you paste in y(t) it is interpreted as y(t)(t):

dsolve(y(t).diff(y), y)
Traceback (most recent call last):
    ...
TypeError: 'y' object is not callable
So remember to exclude the independent variable call (t):

dsolve(y.diff(t), y)
Eq(y(t), C1)
Use the Solution Result
Unlike other solving functions, dsolve() returns an Equality (equation) formatted as, for example, Eq(y(x), C1*sin(3*x) + C2*cos(3*x)) which is equivalent to the mathematical notation 
.

Extract the Result for One Solution and Function
You can extract the result from an Equality using the right-hand side property rhs:

from sympy import Function, dsolve, Derivative
from sympy.abc import x
y = Function('y')
result = dsolve(Derivative(y(x), x, x) + 9*y(x), y(x))
result
Eq(y(x), C1*sin(3*x) + C2*cos(3*x))
result.rhs
C1*sin(3*x) + C2*cos(3*x)
Some ODEs Cannot Be Solved Explicitly, Only Implicitly
The above ODE can be solved explicitly, specifically 
 can be expressed in terms of functions of 
. However, some ODEs cannot be solved explicitly, for example:

from sympy import dsolve, exp, symbols, Function
f = symbols("f", cls=Function)
x = symbols("x")
dsolve(f(x).diff(x) + exp(-f(x))*f(x))
Eq(Ei(f(x)), C1 - x)
This gives no direct expression for 
. Instead, dsolve() expresses a solution as 
 where 
 is Ei, the classical exponential integral function. Ei does not have a known closed-form inverse, so a solution cannot be explicitly expressed as 
 equaling a function of 
. Instead, dsolve returns an implicit solution.

When dsolve returns an implicit solution, extracting the right-hand side of the returned equality will not give an explicitly expression for the function to be solved for, here 
. So before extracting an expression for the function to be solved for, check that dsolve was able to solve for the function explicitly.

Extract the Result for Multiple Function-Solution Pairs
If you are solving a system of equations with multiple unknown functions, the form of the output of dsolve() depends on whether there is one or multiple solutions.

If There is One Solution Set
If there is only one solution set to a system of equations with multiple unknown functions, dsolve() will return a non-nested list containing an equality. You can extract the solution expression using a single loop or comprehension:

from sympy import symbols, Eq, Function, dsolve
y, z = symbols("y z", cls=Function)
x = symbols("x")
eqs_one_soln_set = [Eq(y(x).diff(x), z(x)**2), Eq(z(x).diff(x), z(x))]
solutions_one_soln_set = dsolve(eqs_one_soln_set, [y(x), z(x)])
solutions_one_soln_set
[Eq(y(x), C1 + C2**2*exp(2*x)/2), Eq(z(x), C2*exp(x))]
# Loop through list approach
solution_one_soln_set_dict = {}
for fn in solutions_one_soln_set:
        solution_one_soln_set_dict.update({fn.lhs: fn.rhs})
solution_one_soln_set_dict
{y(x): C1 + C2**2*exp(2*x)/2, z(x): C2*exp(x)}
# List comprehension approach
solution_one_soln_set_dict = {fn.lhs:fn.rhs for fn in solutions_one_soln_set}
solution_one_soln_set_dict
{y(x): C1 + C2**2*exp(2*x)/2, z(x): C2*exp(x)}
# Extract expression for y(x)
solution_one_soln_set_dict[y(x)]
C1 + C2**2*exp(2*x)/2
If There are Multiple Solution Sets
If there are multiple solution sets to a system of equations with multiple unknown functions, dsolve() will return a nested list of equalities, the outer list representing each solution and the inner list representing each function. While you can extract results by specifying the index of each function, we recommend an approach which is robust with respect to function ordering. The following converts each solution into a dictionary so you can easily extract the result for the desired function. It uses standard Python techniques such as loops or comprehensions, in a nested fashion.

from sympy import symbols, Eq, Function, dsolve
y, z = symbols("y z", cls=Function)
x = symbols("x")
eqs = [Eq(y(x).diff(x)**2, z(x)**2), Eq(z(x).diff(x), z(x))]
solutions = dsolve(eqs, [y(x), z(x)])
solutions
[[Eq(y(x), C1 - C2*exp(x)), Eq(z(x), C2*exp(x))], [Eq(y(x), C1 + C2*exp(x)), Eq(z(x), C2*exp(x))]]
# Nested list approach
solutions_list = []
for solution in solutions:
    solution_dict = {}
    for fn in solution:
            solution_dict.update({fn.lhs: fn.rhs})
    solutions_list.append(solution_dict)
solutions_list
[{y(x): C1 - C2*exp(x), z(x): C2*exp(x)}, {y(x): C1 + C2*exp(x), z(x): C2*exp(x)}]
# Nested comprehension approach
solutions_list = [{fn.lhs:fn.rhs for fn in solution} for solution in solutions]
solutions_list
[{y(x): C1 - C2*exp(x), z(x): C2*exp(x)}, {y(x): C1 + C2*exp(x), z(x): C2*exp(x)}]
# Extract expression for y(x)
solutions_list[0][y(x)]
C1 - C2*exp(x)
Work With Arbitrary Constants
You can manipulate arbitrary constants such as C1, C2, and C3, which are generated automatically by dsolve(), by creating them as symbols. For example, if you want to assign values to arbitrary constants, you can create them as symbols and then substitute in their values using subs():

from sympy import Function, dsolve, Derivative, symbols, pi
y = Function('y')
x, C1, C2 = symbols("x, C1, C2")
result = dsolve(Derivative(y(x), x, x) + 9*y(x), y(x)).rhs
result
C1*sin(3*x) + C2*cos(3*x)
result.subs({C1: 7, C2: pi})
7*sin(3*x) + pi*cos(3*x)
Numerically Solve an ODE in SciPy
A common workflow which leverages SciPy’s fast numerical ODE solving is

set up an ODE in SymPy

convert it to a numerical function using lambdify()

solve the initial value problem by numerically integrating the ODE using SciPy’s solve_ivp.

Here is an example from the field of chemical kinetics where the nonlinear ordinary differential equations take this form:

 
 
 
 
and

 
 
from sympy import symbols, lambdify
import numpy as np
import scipy.integrate
import matplotlib.pyplot as plt
# Create symbols y0, y1, and y2
y = symbols('y:3')
kf, kb = symbols('kf kb')
rf = kf * y[0]**2 * y[1]
rb = kb * y[2]**2
# Derivative of the function y(t); values for the three chemical species
# for input values y, kf, and kb
ydot = [2*(rb - rf), rb - rf, 2*(rf - rb)]
ydot
[2*kb*y2**2 - 2*kf*y0**2*y1, kb*y2**2 - kf*y0**2*y1, -2*kb*y2**2 + 2*kf*y0**2*y1]
t = symbols('t') # not used in this case
# Convert the SymPy symbolic expression for ydot into a form that
# SciPy can evaluate numerically, f
f = lambdify((t, y, kf, kb), ydot)
k_vals = np.array([0.42, 0.17]) # arbitrary in this case
y0 = [1, 1, 0] # initial condition (initial values)
t_eval = np.linspace(0, 10, 50) # evaluate integral from t = 0-10 for 50 points
# Call SciPy's ODE initial value problem solver solve_ivp by passing it
#   the function f,
#   the interval of integration,
#   the initial state, and
#   the arguments to pass to the function f
solution = scipy.integrate.solve_ivp(f, (0, 10), y0, t_eval=t_eval, args=k_vals)
# Extract the y (concentration) values from SciPy solution result
y = solution.y
# Plot the result graphically using matplotlib
plt.plot(t_eval, y.T)
# Add title, legend, and axis labels to the plot
plt.title('Chemical Kinetics')
plt.legend(['NO', 'Br$_2$', 'NOBr'], shadow=True)
plt.xlabel('time')
plt.ylabel('concentration')
# Finally, display the annotated plot
plt.show()
(png, hires.png, pdf)

../../_images/solve-ode-1.png
SciPy’s solve_ivp returns a result containing y (numerical function result, here, concentration) values for each of the three chemical species, corresponding to the time points t_eval.

Ordinary Differential Equation Solving Hints
Return Unevaluated Integrals
By default, dsolve() attempts to evaluate the integrals it produces to solve your ordinary differential equation. You can disable evaluation of the integrals by using Hint Functions ending with _Integral, for example separable_Integral. This is useful because integrate() is an expensive routine. SymPy may hang (appear to never complete the operation) because of a difficult or impossible integral, so using an _Integral hint will at least return an (unintegrated) result, which you can then consider. The simplest way to disable integration is with the all_Integral hint because you do not need to know which hint to supply: for any hint with a corresponding _Integral hint, all_Integral only returns the _Integral hint.

Select a Specific Solver
You may wish to select a specific solver using a hint for a couple of reasons:

educational purposes: for example if you are learning about a specific method to solve ODEs and want to get a result that exactly matches that method

form of the result: sometimes an ODE can be solved by many different solvers, and they can return different results. They will be mathematically equivalent, though the arbitrary constants may not be. dsolve() by default tries to use the “best” solvers first, which are most likely to return the most usable output, but it is not a perfect heuristic. For example, the “best” solver may produce a result with an integral that SymPy cannot solve, but another solver may produce a different integral that SymPy can solve. So if the solution isn’t in a form you like, you can try other hints to check whether they give a preferable result.

Not All Equations Can Be Solved
Equations With No Solution
Not all differential equations can be solved, for example:

from sympy import Function, dsolve, Derivative, symbols
y = Function('y')
x, C1, C2 = symbols("x, C1, C2")
dsolve(Derivative(y(x), x, 3) - (y(x)**2), y(x)).rhs
Traceback (most recent call last):
    ...
NotImplementedError: solve: Cannot solve -y(x)**2 + Derivative(y(x), (x, 3))
Equations With No Closed-Form Solution
As noted above, Some ODEs Cannot Be Solved Explicitly, Only Implicitly.

Also, some systems of differential equations have no closed-form solution because they are chaotic, for example the Lorenz system or a double pendulum described by these two differential equations (simplified from ScienceWorld):

from sympy import symbols, Function, cos, sin, dsolve
theta1, theta2 = symbols('theta1 theta2', cls=Function)
g, t = symbols('g t')
eq1 = 2*theta1(t).diff(t, t) + theta2(t).diff(t, t)*cos(theta1(t) - theta2(t)) + theta2(t).diff(t)**2*sin(theta1(t) - theta2(t)) + 2*g*sin(theta1(t))
eq2 = theta2(t).diff(t, t) + theta1(t).diff(t, t)*cos(theta1(t) - theta2(t)) - theta1(t).diff(t)**2*sin(theta1(t) - theta2(t)) + g*sin(theta2(t))
dsolve([eq1, eq2], [theta1(t), theta2(t)])
Traceback (most recent call last):
...
NotImplementedError